# 1:N Multi-Project Deployment

Deploy **multiple team projects** under a **single shared AI Foundry Account** -- the 1:N pattern.

## What Gets Deployed

| Resource | Purpose |
|----------|--------|
| AI Foundry Account (1) | Shared departmental account |
| Foundry Projects (N) | One workspace per team (beta, delta, gamma) |
| APIM Connections (N) | Each project gets its own gateway connection to landing zone models |

## Why 1:N?

In enterprise environments, teams within the same department share infrastructure costs while maintaining **project-level isolation**:

- **Cost efficiency** -- one AI account instead of N accounts
- **Isolation** -- each team has its own project workspace, agents, and data
- **Shared RBAC** -- account-level roles propagate; project-level roles scope access
- **Unified management** -- single account to monitor, patch, and govern

> Prerequisite: Complete **04-02** first to deploy the Landing Zone

## Step 1: Load Core Configuration

In [1]:
import os, subprocess
from pathlib import Path

# Load .env from repo root
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

CORE_ENDPOINT = os.environ['CORE_ENDPOINT']
GATEWAY_URL = os.environ['GATEWAY_URL']
GATEWAY_KEY = os.environ['ALPHA_GATEWAY_KEY']
CHAT_MODEL = os.environ['CHAT_MODEL']

print(f"Core Endpoint: {CORE_ENDPOINT}")
print(f"Gateway URL:  {GATEWAY_URL}")
print(f"Gateway Key:  {GATEWAY_KEY[:2]}... (hidden)")
print(f"Chat Model:   {CHAT_MODEL}")

Core Endpoint: https://aif-core-c2676f.cognitiveservices.azure.com/
Gateway URL:  https://apim-foundry-c2676f.azure-api.net/openai
Gateway Key:  83... (hidden)
Chat Model:   gpt-4.1-mini


## Step 2: Set Variables

In [2]:
import hashlib, subprocess

# Derive a stable 6-char suffix from the subscription ID — same as Lab 1A
SUB_ID = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]

MULTI_RG = f"rg-foundry-multi-{SUFFIX}"
LOCATION = "eastus2"
TEAM_NAMES = ["beta", "delta", "gamma"]
PROJECT_COUNT = len(TEAM_NAMES)

print(f"Suffix:         {SUFFIX}")
print(f"Resource Group: {MULTI_RG}")
print(f"Teams:          {', '.join(TEAM_NAMES)}")
print(f"Project Count:  {PROJECT_COUNT}")

Suffix:         c2676f
Resource Group: rg-foundry-multi-c2676f
Teams:          beta, delta, gamma
Project Count:  3


## Step 3: Create Resource Group

In [3]:
!az group create -n "{MULTI_RG}" -l "{LOCATION}" -o table

Location    Name
----------  -----------------------
eastus2     rg-foundry-multi-c2676f


## Step 4: Deploy 1:N Infrastructure

Deploys one AI Foundry Account with N projects and their APIM connections.

Takes ~2-3 minutes

In [4]:
import subprocess, json, base64

# Get principal ID from cached JWT token (avoids graph.microsoft.com network call)
token = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload = token.split('.')[1] + '=='
PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']
print(f"Principal ID: {PRINCIPAL_ID}")

team_names_json = json.dumps(TEAM_NAMES)

# Use subprocess.run to avoid shell quoting issues with JSON arrays
result = subprocess.run(
    [
        "az", "deployment", "group", "create",
        "-g", MULTI_RG,
        "--template-file", "main.bicep",
        "-p", f"deployerPrincipalId={PRINCIPAL_ID}",
        "-p", f"apimUrl={GATEWAY_URL}",
        "-p", f"modelName={CHAT_MODEL}",
        "-p", f"apimSubscriptionKey={GATEWAY_KEY}",
        "-p", f"projectCount={PROJECT_COUNT}",
        "-p", f"teamNames={team_names_json}",
        "-p", f"suffix={SUFFIX}",
        "-o", "table",
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Principal ID: 5db0aa5d-f281-47a3-9720-04727dec61e8
Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  -----------------------
main    Succeeded  2026-05-10T11:55:37.465024+00:00  Incremental  rg-foundry-multi-c2676f



## Step 5: Get Outputs

In [5]:
import subprocess, json
from pathlib import Path

r = subprocess.run(
    f'az deployment group show -g "{MULTI_RG}" -n main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)

if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(
        f"Failed to retrieve deployment outputs. "
        f"Make sure the deployment in Step 4 succeeded.\n{r.stderr}"
    )

out = json.loads(r.stdout)

ACCOUNT_NAME = out['accountName']['value']
ACCOUNT_ENDPOINT = out['accountEndpoint']['value']
PROJECT_NAMES = out['projectNames']['value']
PROJECT_ENDPOINTS = out['projectEndpoints']['value']
APIM_CONNECTIONS = out['apimConnectionNames']['value']

print(f"Shared Account:    {ACCOUNT_NAME}")
print(f"Account Endpoint:  {ACCOUNT_ENDPOINT}")
print()
for i, (name, endpoint) in enumerate(zip(PROJECT_NAMES, PROJECT_ENDPOINTS)):
    print(f"  Project {i+1} ({TEAM_NAMES[i]}): {name}")
    print(f"    Endpoint:   {endpoint}")
    print(f"    Connection: {APIM_CONNECTIONS[i]}")

# Merge outputs into .env file in repo root (preserves existing keys)
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'

existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

# Remove stale keys (renamed in prior versions of this notebook)
for team in TEAM_NAMES:
    existing.pop(f'{team.upper()}_FOUNDRY_GATEWAY_CONNECTION', None)

new_values = {
    'MULTI_ACCOUNT': ACCOUNT_NAME,
    'MULTI_ACCOUNT_ENDPOINT': ACCOUNT_ENDPOINT,
}
for i, name in enumerate(PROJECT_NAMES):
    team = TEAM_NAMES[i].upper()
    new_values[f'{team}_FOUNDRY_PROJECT'] = name
    new_values[f'{team}_FOUNDRY_PROJECT_ENDPOINT'] = PROJECT_ENDPOINTS[i]
    new_values[f'{team}_FOUNDRY_CORE_CONNECTION'] = APIM_CONNECTIONS[i]

existing.update(new_values)
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')

print(f"\n✅ Outputs saved to {env_file}")

Shared Account:    aif-spoke-multi-c2676f
Account Endpoint:  https://aif-spoke-multi-c2676f.cognitiveservices.azure.com/

  Project 1 (beta): project-beta-c2676f
    Endpoint:   https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-beta-c2676f
    Connection: core-beta
  Project 2 (delta): project-delta-c2676f
    Endpoint:   https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-delta-c2676f
    Connection: core-delta
  Project 3 (gamma): project-gamma-c2676f
    Endpoint:   https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/project-gamma-c2676f
    Connection: core-gamma

✅ Outputs saved to /home/jp/development/corticalstack/foundry-nextgen/.env


## Step 5b: Inspect Project Connections

List the APIM connection registered on each project. Note that `credentials` only shows the auth type — the actual API key is stored securely by Foundry and never returned.

In [6]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
import json

credential = DefaultAzureCredential()

for i, endpoint in enumerate(PROJECT_ENDPOINTS):
    team = TEAM_NAMES[i]
    client = AIProjectClient(credential=credential, endpoint=endpoint)
    connections = list(client.connections.list())
    print(f"Project: {PROJECT_NAMES[i]} (team: {team})")
    for c in connections:
        d = dict(c)
        models = json.loads(d.get('metadata', {}).get('models', '[]'))
        print(f"  name:              {d.get('name')}")
        print(f"  type:              {d.get('type')}")
        print(f"  target:            {d.get('target')}")
        print(f"  isDefault:         {d.get('isDefault')}")
        print(f"  credentials:       {d.get('credentials')}")
        print(f"  inferenceVersion:  {d.get('metadata', {}).get('inferenceAPIVersion')}")
        print(f"  models:            {[m['name'] for m in models]}")
    print()

Project: project-beta-c2676f (team: beta)
  name:              core-beta
  type:              ApiManagement
  target:            https://apim-foundry-c2676f.azure-api.net/openai
  isDefault:         True
  credentials:       {'type': 'ApiKey'}
  inferenceVersion:  2024-10-21
  models:            ['gpt-4.1-mini']

Project: project-delta-c2676f (team: delta)
  name:              core-delta
  type:              ApiManagement
  target:            https://apim-foundry-c2676f.azure-api.net/openai
  isDefault:         False
  credentials:       {'type': 'ApiKey'}
  inferenceVersion:  2024-10-21
  models:            ['gpt-4.1-mini']

Project: project-gamma-c2676f (team: gamma)
  name:              core-gamma
  type:              ApiManagement
  target:            https://apim-foundry-c2676f.azure-api.net/openai
  isDefault:         False
  credentials:       {'type': 'ApiKey'}
  inferenceVersion:  2024-10-21
  models:            ['gpt-4.1-mini']



## Step 6: Create Per-Team APIM Hub Subscriptions

Each team gets its own APIM hub subscription so rate limiting and usage tracking are scoped per team.
The per-team key is then patched into that team's Foundry project connection.

In [7]:
import subprocess, json, tempfile
from pathlib import Path

sub_id = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
CORE_RG = f"rg-foundry-core-{SUFFIX}"
APIM_NAME = f"apim-foundry-{SUFFIX}"

team_keys = {}

for i, team in enumerate(TEAM_NAMES):
    sub_name = f"foundry-gateway-{team}"
    apim_base = (
        f"https://management.azure.com/subscriptions/{sub_id}"
        f"/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"
    )

    # Create (or update) per-team APIM subscription scoped to the openai API
    create_result = subprocess.run(
        [
            "az", "rest", "--method", "PUT",
            "--uri", f"{apim_base}/subscriptions/{sub_name}?api-version=2024-06-01-preview",
            "--body", json.dumps({
                "properties": {
                    "displayName": f"Team {team.capitalize()} Gateway Access",
                    "scope": (
                        f"/subscriptions/{sub_id}/resourceGroups/{CORE_RG}"
                        f"/providers/Microsoft.ApiManagement/service/{APIM_NAME}/apis/openai"
                    ),
                    "state": "active"
                }
            }),
            "--headers", "Content-Type=application/json",
        ],
        capture_output=True, text=True
    )
    if create_result.returncode != 0:
        print(f"❌ {team}: subscription create failed — {create_result.stderr.strip()}")
        continue

    # Retrieve the primary key for this team
    key = subprocess.run(
        f'az rest --method POST'
        f' --uri "{apim_base}/subscriptions/{sub_name}/listSecrets?api-version=2024-06-01-preview"'
        f' --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    team_keys[team] = key

    # PATCH the team's Foundry project connection to use the per-team key
    models_list = [{"name": CHAT_MODEL, "properties": {"model": {"name": CHAT_MODEL, "version": "", "format": "OpenAI"}}}]
    connection_uri = (
        f"https://management.azure.com/subscriptions/{sub_id}"
        f"/resourceGroups/{MULTI_RG}/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}"
        f"/projects/{PROJECT_NAMES[i]}/connections/{APIM_CONNECTIONS[i]}?api-version=2025-04-01-preview"
    )
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump({
            "properties": {
                "category": "ApiManagement",
                "target": GATEWAY_URL,
                "authType": "ApiKey",
                "credentials": {"key": key},
                "metadata": {
                    "deploymentInPath": "true",
                    "inferenceAPIVersion": "2024-10-21",
                    "models": json.dumps(models_list),
                },
            }
        }, f)
        payload_file = f.name

    patch_result = subprocess.run(
        f'az rest --method PATCH --uri "{connection_uri}"'
        f' --body @"{payload_file}" --headers "Content-Type=application/json" -o json',
        shell=True, capture_output=True, text=True
    )
    status = "✅" if patch_result.returncode == 0 else "❌"
    print(f"{status} Team {team}: key {key[:2]}... (hidden), connection updated ({APIM_CONNECTIONS[i]})")

# Persist per-team keys to .env
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'
existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

for team, key in team_keys.items():
    existing[f'{team.upper()}_GATEWAY_KEY'] = key

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f"\n✅ .env updated: {', '.join(f'{t.upper()}_GATEWAY_KEY' for t in team_keys)}")

✅ Team beta: key 23... (hidden), connection updated (core-beta)
✅ Team delta: key 73... (hidden), connection updated (core-delta)
✅ Team gamma: key 43... (hidden), connection updated (core-gamma)

✅ .env updated: BETA_GATEWAY_KEY, DELTA_GATEWAY_KEY, GAMMA_GATEWAY_KEY


## Step 7: Test Each Project

Connect to every project independently and verify each can reach the landing zone models through its own APIM connection.

> APIM gateway connections require the **Agent + Responses API** path; `chat.completions` is not supported for this connection type.

Dependencies are managed via pyproject.toml — run `uv sync` in the repo root if packages are missing.

In [8]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

credential = DefaultAzureCredential()

print(f"Testing {len(PROJECT_ENDPOINTS)} projects...")
print("=" * 60)

for i, endpoint in enumerate(PROJECT_ENDPOINTS):
    team = TEAM_NAMES[i]
    project_name = PROJECT_NAMES[i]
    gateway_model = f"{APIM_CONNECTIONS[i]}/{CHAT_MODEL}"

    client = AIProjectClient(credential=credential, endpoint=endpoint)

    # APIM connections require the Agent + Responses API (chat.completions won't work)
    agent = client.agents.create_version(
        agent_name=f"verify-{team}",
        definition=PromptAgentDefinition(
            model=gateway_model,
            instructions="Reply in one sentence only.",
        ),
    )

    openai_client = client.get_openai_client()
    response = openai_client.responses.create(
        input=f"Reply in one sentence: which team are you serving? Team {team}.",
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference",
            }
        },
    )
    answer = response.output_text if hasattr(response, "output_text") else str(response.output)

    # Cleanup verification agent
    client.agents.delete(agent_name=agent.name)

    print(f"\nProject {i+1} ({team}): {project_name}")
    print(f"  Gateway model: {gateway_model}")
    print(f"  Response: {answer}")

print("\n" + "=" * 60)
print(f"All {len(PROJECT_ENDPOINTS)} projects verified.")

Testing 3 projects...

Project 1 (beta): project-beta-c2676f
  Gateway model: core-beta/gpt-4.1-mini
  Response: I am serving Team Beta.

Project 2 (delta): project-delta-c2676f
  Gateway model: core-delta/gpt-4.1-mini
  Response: I am serving Team Delta.

Project 3 (gamma): project-gamma-c2676f
  Gateway model: core-gamma/gpt-4.1-mini
  Response: I am here to assist Team Gamma with anything you need!

All 3 projects verified.


## Step 8: Create Per-Team Agents

Each project gets its own agent -- demonstrating that agents are scoped to projects, not accounts.

In [9]:
from azure.ai.projects.models import PromptAgentDefinition

agents = []

for i, endpoint in enumerate(PROJECT_ENDPOINTS):
    team = TEAM_NAMES[i]
    gateway_model = f"{APIM_CONNECTIONS[i]}/{CHAT_MODEL}"
    client = AIProjectClient(credential=credential, endpoint=endpoint)

    agent = client.agents.create_version(
        agent_name=f"team-{team}-agent",
        definition=PromptAgentDefinition(
            model=gateway_model,
            instructions=(
                f"You are the dedicated assistant for Team {team.capitalize()}. "
                f"Always identify yourself as the Team {team.capitalize()} assistant. "
                "Keep responses brief."
            ),
        ),
    )
    agents.append((team, agent, client))
    print(f"Agent created: {agent.name} v{agent.version} (project: {PROJECT_NAMES[i]}, model: {gateway_model})")

print(f"\n{len(agents)} team agents created across {len(PROJECT_ENDPOINTS)} projects.")

Agent created: team-beta-agent v1 (project: project-beta-c2676f, model: core-beta/gpt-4.1-mini)
Agent created: team-delta-agent v1 (project: project-delta-c2676f, model: core-delta/gpt-4.1-mini)
Agent created: team-gamma-agent v1 (project: project-gamma-c2676f, model: core-gamma/gpt-4.1-mini)

3 team agents created across 3 projects.


In [10]:
# Invoke each agent
for team, agent, client in agents:
    openai_client = client.get_openai_client()
    response = openai_client.responses.create(
        input=f"Hello! Which team do you belong to?",
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference"
            }
        }
    )
    answer = response.output_text if hasattr(response, 'output_text') else str(response.output)
    print(f"Team {team}: {answer}")
    print()

Team beta: Hello! I am the Team Beta assistant. How can I help you today?

Team delta: Hello! I am the Team Delta assistant. How can I help you today?

Team gamma: Hello! I am the Team Gamma assistant. How can I help you today?



## Done!

You deployed the **1:N pattern**: one AI Foundry Account hosting multiple team projects.

### Key Concepts

| Concept | Description |
|---------|-------------|
| 1:N Pattern | One AI Account, many projects -- shared infra with project-level isolation |
| Bicep loop | `for i in range(0, projectCount)` creates N projects and connections |
| Agent scoping | Agents live inside a project; Team Beta cannot see Team Delta's agents |
| Shared RBAC | Account-level `Cognitive Services User` role applies to all projects |
| Per-project connections | Each project has its own uniquely-named APIM connection (e.g., `core-beta`) |
| Per-team APIM keys | Each team has its own APIM subscription — independent rate limiting and usage tracking (`BETA_GATEWAY_KEY`, `DELTA_GATEWAY_KEY`, etc.) |

### When to use 1:N vs. separate accounts

| Scenario | Recommendation |
|----------|----------------|
| Teams in the same department / cost center | 1:N (this lab) |
| Teams with different compliance boundaries | Separate accounts (Lab 1B) |
| Dev / staging / prod environments | Separate accounts per environment |
| Rapid prototyping with many small teams | 1:N to avoid account sprawl |

## Cleanup (Optional)

In [11]:
# for team, agent, client in agents:
#     client.agents.delete(agent_name=agent.name)
#     print(f"Deleted agent: {agent.name}")
# !az group delete -n "{MULTI_RG}" --yes --no-wait

---

## Troubleshooting: IfMatchPreconditionFailed on Redeploy

When redeploying after a prior successful run, ARM's incremental mode does a conditional PUT on existing connections using their stored ETag. If any connection was modified since the last deployment (e.g. an agent interacted with it), the ETag will no longer match and the deployment fails:

```
IfMatchPreconditionFailed: The specified precondition 'If-Match = "..."' failed.
```

**Fix:** Delete the APIM connections before redeploying. They will be recreated by the bicep. Run the cell below, then re-run Step 4.

In [12]:
import subprocess, json

sub_id = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()

r = subprocess.run(
    f'az deployment group show -g "{MULTI_RG}" -n main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)

if r.returncode != 0 or not r.stdout.strip():
    print("No previous deployment found — nothing to clean up")
else:
    out = json.loads(r.stdout)
    account = out['accountName']['value']
    project_names = out['projectNames']['value']
    connection_names = out['apimConnectionNames']['value']

    for project, conn in zip(project_names, connection_names):
        uri = (
            f"https://management.azure.com/subscriptions/{sub_id}"
            f"/resourceGroups/{MULTI_RG}/providers/Microsoft.CognitiveServices/accounts/{account}"
            f"/projects/{project}/connections/{conn}?api-version=2025-04-01-preview"
        )
        result = subprocess.run(
            f'az rest --method DELETE --uri "{uri}"',
            shell=True, capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✅ Deleted connection: {conn} (project: {project})")
        else:
            print(f"⚠️  {conn}: {result.stderr.strip()}")

    print("\nConnections purged — re-run Step 4 to redeploy.")

✅ Deleted connection: core-beta (project: project-beta-c2676f)
✅ Deleted connection: core-delta (project: project-delta-c2676f)
✅ Deleted connection: core-gamma (project: project-gamma-c2676f)

Connections purged — re-run Step 4 to redeploy.


### Troubleshooting: Soft-Deleted Foundry Account

If you delete the resource group and try to redeploy with the same subscription, the Foundry account is soft-deleted and ARM will refuse to recreate it:

```
FlagMustBeSetForRestore: An existing resource ... has been soft-deleted.
```

List and purge it before redeploying:

```bash
az cognitiveservices account list-deleted -o table
az cognitiveservices account purge -l eastus2 -g "{MULTI_RG}" -n "foundry-multi-{SUFFIX_FROM_BICEP}"
```

The bicep-generated suffix for the account name is derived from `uniqueString(subscriptionId, resourceGroupId)` and will differ from the notebook's `SUFFIX` variable. Read the exact name from `az cognitiveservices account list-deleted`.